# Episode 30: Redundancy & Voting

A single LLM call can be wrong — a misread, an unlucky sample, a hallucinated detail. For decisions that matter, asking *once* isn't enough.

**In this episode you'll build:** a review panel — several agents answer the *same* question independently and in parallel, and a coordinator takes the consensus.

This opens **Group 7 — Advanced**.

## Why redundancy

The idea is borrowed from engineering: when one component can fail, you run several and vote. For agents:

- **Independent attempts** — each worker answers fresh, with no knowledge of the others. One bad sample doesn't sink the result.
- **Parallel** — the workers run concurrently, so three opinions cost roughly the *latency* of one.
- **Consensus** — the coordinator reconciles the answers (majority vote, or synthesis).

It costs more tokens than a single call. Spend it where being wrong is expensive.

## The panel pattern

`TaskConfig` (Episode 4 / the Agent Harness) gives a coordinator the auto-injected `run_subtasks` tool. `run_subtasks(tasks=[...], parallel=True)` fans out a list of tasks to fresh worker agents **concurrently**.

Here the coordinator sends the *same* question three times, so three independent workers answer in parallel — then it reports the consensus. The observer shows the fan-out as it happens.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import json

from autogen.beta import Agent, TaskConfig, observer
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import ToolCallEvent

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


@observer(ToolCallEvent)
def show_fanout(event: ToolCallEvent) -> None:
    if event.name == "run_subtasks":
        tasks = json.loads(event.arguments).get("tasks", [])
        print(f"[fan-out] run_subtasks dispatched {len(tasks)} workers in parallel")


panel_lead = Agent(
    "panel_lead",
    prompt=(
        "You run a 3-member review panel. Call run_subtasks ONCE with the question "
        "worded three times, so three workers answer it independently and in parallel. "
        "Then report the majority verdict as 'CONSENSUS: <one-word answer>'."
    ),
    config=config,
    tasks=TaskConfig(prompt="Answer with one word only: SAFE or UNSAFE."),
    observers=[show_fanout],
)

reply = await panel_lead.ask(
    "Is it safe to run `rm -rf /` as root on a production server?"
)
print("RESULT:", reply.body)

## Voting peers vs. a coordinator

The panel above has a **coordinator** that dispatches and tallies. That's the right shape when one agent owns the decision.

When you want the workers to be true *peers* — each sees the others' answers and they argue toward agreement — use the **Hub discussion adapter** (Episode 13) instead:

```python
channel = await alice.open(
    type="discussion",
    target=[bob.agent_id, carol.agent_id],
    knobs={"ordering": ORDERING_ROUND_ROBIN},
)
await channel.send("Vote: should we ship this release? Justify in one line.")
```

- **`run_subtasks` panel** — workers are isolated; cheap, fast, no cross-talk. Best for independent sampling.
- **Discussion adapter** — workers see each other; they can change their minds. Best when deliberation improves the answer.

## Additional content

**Sequential when dependent.** `run_subtasks` defaults to `parallel=True`. Pass `parallel=False` only when a later task needs an earlier task's result — redundancy *wants* parallel and independent.

**More than majority vote.** The coordinator is just an LLM — it can do more than count. Ask it to synthesize the best parts of each answer, flag disagreement, or escalate to a human when the workers split.

**Diversity helps.** Identical workers asked an identical question mostly agree. For real robustness, vary the workers — different prompts, different `TaskConfig` models, different framings — so their mistakes aren't correlated.

## What's next

Redundancy makes answers *reliable*. Episode 31 makes them *transparent* — **reasoning**: getting an agent to show its work as structured, inspectable output.